In [1]:
import pandas as pd
import numpy as np

DATA DESCRIPTION
```
file name -> Columns
quater-i.csv -> ['order_id', 'quantity', 'item_id', 'choice_description_id' 'item_price']
items.csv -> ['item_id', 'item_name']
```
Dataset Link - https://drive.google.com/drive/folders/1Z0kaFybvgFeczeUj4dldUnhTdloLqLsL?usp=share_link

In [3]:
# import like this
items_path = "items.csv"
q1_path = "quarter-1.csv"
q2_path = "quarter-2.csv"
q3_path = "quarter-3.csv"


q1= pd.read_csv(q1_path)
q2 = pd.read_csv(q2_path)
q3 = pd.read_csv(q3_path)

items = pd.read_csv(items_path)

###`Q:1-5`
1. You are given three quater files, your job is to append these three files and make a single dataframe.
2. Have a index as Q-1 Q-2 Q-3 for respective quater files in the dataframe
3. Your are given a file items.csv which has item_id and item_name. Find out most sold items in each quarter.
4. Find out items which has made most revenue in each quarter.
5. Find out avg order price of each quarter.

***Note: item_price is given as str with $ sign, in earlier task you have converted this to rupees, here too first convert item_price field in rupees.***

In [10]:
# code here
pd.concat([q1,q2,q3],ignore_index=True).head(2)

,order_id,quantity,item_id,choice_description_id,item_price
0,1,1,1,1,$3.39
1,1,1,2,2,$3.39


In [47]:
q=pd.concat([q1,q2,q3],keys=["q1","q2","q3"]).reset_index()
q["item_price"]=q["item_price"].str.strip("$").astype("float16")
q.head(2)

,level_0,level_1,order_id,quantity,item_id,choice_description_id,item_price
0,q1,0,1,1,1,1,3.390625
1,q1,1,1,1,2,2,3.390625


In [28]:
q3

,order_id,quantity,item_id,choice_description_id,item_price


In [52]:
q.merge(items,how="inner",on="item_id").groupby(["level_0","item_name"]).agg(item_count=("item_price","count")).reset_index().sort_values("item_count",ascending=False).drop_duplicates(subset=["level_0"],keep="first")

,level_0,item_name,item_count
65,q2,Chicken Bowl,375
17,q1,Chicken Bowl,351


In [54]:
q.merge(items,how="inner",on="item_id").groupby(["level_0","item_name"]).agg(revenue=("item_price","sum")).reset_index().sort_values("revenue",ascending=False).drop_duplicates(subset=["level_0"],keep="first")

,level_0,item_name,revenue
65,q2,Chicken Bowl,3817.023438
17,q1,Chicken Bowl,3525.773438


In [57]:
q.merge(items,how="inner",on="item_id").groupby("level_0").agg(Avg_revenue=("item_price","mean"))

,Avg_revenue
level_0,
q1,7.577199
q2,7.355066


###`Q-6` From the IPL wala dataset you have to find the Purple cap holder each season.

*Note: Bowler with most no wickets in a season gets purple cap. If more than one bowler have same no of wickets in the season, one with least ecomnomy among them is purple cap holder.*

Bowler's Economy = runs-conceded per six balls

In [101]:
# code here
balls=pd.read_csv("IPL_Ball_by_Ball_2008_2022.csv")
balls["extra_type"].unique()

array([nan, 'legbyes', 'wides', 'byes', 'noballs', 'penalty'],
      dtype=object)

In [61]:
ipl=pd.read_csv("IPL_Matches_2008_2022.csv")
ipl.head(2)

,ID,City,Date,Season,MatchNumber,Team1,Team2,Venue,TossWinner,TossDecision,SuperOver,WinningTeam,WonBy,Margin,method,Player_of_Match,Team1Players,Team2Players,Umpire1,Umpire2
0,1312200,Ahmedabad,2022-05-29,2022,Final,Rajasthan Royals,Gujarat Titans,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,bat,N,Gujarat Titans,Wickets,7.0,NaN,HH Pandya,"['YBK Jaiswal', 'JC Buttler', 'SV Samson', 'D ...","['WP Saha', 'Shubman Gill', 'MS Wade', 'HH Pan...",CB Gaffaney,Nitin Menon
1,1312199,Ahmedabad,2022-05-27,2022,Qualifier 2,Royal Challengers Bangalore,Rajasthan Royals,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,field,N,Rajasthan Royals,Wickets,7.0,NaN,JC Buttler,"['V Kohli', 'F du Plessis', 'RM Patidar', 'GJ ...","['YBK Jaiswal', 'JC Buttler', 'SV Samson', 'D ...",CB Gaffaney,Nitin Menon


In [69]:
df.head(2)

,ID,City,Date,Season,MatchNumber,Team1,Team2,Venue,TossWinner,TossDecision,...,extra_type,batsman_run,extras_run,total_run,non_boundary,isWicketDelivery,player_out,kind,fielders_involved,BattingTeam
0,1312200,Ahmedabad,2022-05-29,2022,Final,Rajasthan Royals,Gujarat Titans,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,bat,...,NaN,0,0,0,0,0,NaN,NaN,NaN,Rajasthan Royals
1,1312200,Ahmedabad,2022-05-29,2022,Final,Rajasthan Royals,Gujarat Titans,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,bat,...,legbyes,0,1,1,0,0,NaN,NaN,NaN,Rajasthan Royals


In [81]:
df=ipl.merge(balls,how="inner",left_on="ID",right_on="ID")
temp=df.groupby(["Season","bowler"]).agg(wickets=("isWicketDelivery","sum"),runs=("total_run",sum),balls=("total_run","count")).reset_index()
temp['economy']=temp["runs"]/((temp["balls"])/6)
temp.sort_values(["wickets","economy"],ascending=[False,True]).drop_duplicates(subset=["Season"]).sort_values("Season")


C:\Users\jagan\AppData\Local\Temp\ipykernel_26396\3521222641.py:2: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  temp=df.groupby(["Season","bowler"]).agg(wickets=("isWicketDelivery","sum"),runs=("total_run",sum),balls=("total_run","count")).reset_index()


,Season,bowler,wickets,runs,balls,economy
84,2007/08,Sohail Tanvir,24,275,265,6.226415
174,2009,RP Singh,26,430,382,6.753927
284,2009/10,PP Ojha,22,439,360,7.316667
447,2011,SL Malinga,30,393,397,5.939547
537,2012,M Morkel,30,466,389,7.187661
629,2013,DJ Bravo,34,505,392,7.729592
779,2014,MM Sharma,26,457,324,8.462963
847,2015,DJ Bravo,28,442,324,8.185185
938,2016,B Kumar,24,503,414,7.289855
1048,2017,B Kumar,28,384,324,7.111111


###`Q-7:` Best bowler in death overs. 
*Note: Have taken most no of wickets in case of tie with least economy*

Death Overs - [16-20]

In [86]:
df=ipl.merge(balls,how="inner",left_on="ID",right_on="ID")
df=df[df["overs"] >= 16]
temp=df.groupby(["Season","bowler"]).agg(wickets=("isWicketDelivery","sum"),runs=("total_run",sum),balls=("total_run","count")).reset_index()
temp['economy']=temp["runs"]/((temp["balls"])/6)
temp.sort_values(["wickets","economy"],ascending=[False,True]).drop_duplicates(subset=["Season"]).sort_values("Season")


C:\Users\jagan\AppData\Local\Temp\ipykernel_26396\3854612242.py:3: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  temp=df.groupby(["Season","bowler"]).agg(wickets=("isWicketDelivery","sum"),runs=("total_run",sum),balls=("total_run","count")).reset_index()


,Season,bowler,wickets,runs,balls,economy
63,2007/08,Sohail Tanvir,18,110,98,6.734694
135,2009,RP Singh,15,153,125,7.344000
190,2009/10,IK Pathan,10,159,85,11.223529
275,2011,DE Bollinger,13,132,125,6.336000
425,2012,SP Narine,21,165,140,7.071429
458,2013,DJ Bravo,21,217,169,7.704142
567,2014,MM Sharma,15,210,119,10.588235
612,2015,DJ Bravo,22,258,176,8.795455
711,2016,MM Sharma,13,161,110,8.781818
771,2017,JD Unadkat,18,155,108,8.611111


###`Q-8` Batsman record season wise

Make a function which takes a input `batsman_name` and it returns a dataframe.
Columns of the data frame are - `['Season','Innings', 'TotalRuns', 'Avg', 'HighestScore','StrikeRate']`.
* In result make `Season` column as index.

* Avg - total_runs/ no of time got out. - player_out column will help.
* StrikeRate -(total_runs/ balls faced) * 100- wides are not included in batsman ball faced counts. No balls are included. -> Extra_type column will help
* Batsman Can score runs on No Balls.
* Batsman can get out on No Ball or Wides. And even while being on non-striker. Keep these things in mind before masking.

In [118]:
# code here
def func(name):
    df=ipl.merge(balls,how="inner",left_on="ID",right_on="ID")
    df=df[(df["batter"]=="V Kohli") & ~(df["extra_type"].isin(['legbyes', 'wides', 'byes', 'penalty']))]
    highest_runs=df.groupby(["Season","ID"]).agg(runs=("batsman_run","sum")).reset_index().sort_values("runs",ascending=False).drop_duplicates(subset=["Season"]).sort_values("Season").set_index("Season")["runs"]
    df=df.groupby("Season").agg(innings=("ID","nunique"),runs=("batsman_run","sum"),balls=("batsman_run","count"))
    df["avg"]=df["runs"]/df["innings"]
    df["highest_runs"]=highest_runs
    df["SR"]=(df["runs"]/df["balls"])*100
    return df



In [119]:
func("V Kohli")

,innings,runs,balls,avg,highest_runs,SR
Season,,,,,,
2007/08,12,165,153,13.750000,38,107.843137
2009,13,246,216,18.923077,50,113.888889
2009/10,13,307,211,23.615385,58,145.497630
2011,16,557,452,34.812500,71,123.230088
2012,15,364,324,24.266667,73,112.345679
2013,16,639,455,39.937500,99,140.439560
2014,14,359,290,25.642857,73,123.793103
2015,16,505,386,31.562500,82,130.829016
2016,16,973,637,60.812500,113,152.747253


###`Q-9` Using both dataset, make a dataframe as described below

Data Frame columns-> `['PlayerOfThematch', 'BattingFigure', 'BowlingFigure']`

* BattingFigure->`<runs>/<balls>`
* BowlingFigure->`<wicket>/<runs-conceded>`

DataFrame should have one record for each match.

Say 'V Kohli' got POM award then in dataset include his batting figure of that match. Say he scored 112runs in 76 balls. And he hasn't bowled so Bowling Figure will be NaN
```
PlayerOfThematch BattingFigure BowlingFigure
V Kohli          112/76         nan  

```


In [ ]:
# code here

## **Questions Based on Iris Dataset**

- **Sepal All:** https://docs.google.com/spreadsheets/d/e/2PACX-1vT58ekmHTwptX7Bs4QOy6YByA1HMvYTACeeIjrKhHE0Pg1K_3egewHMKMh02zN9D5-yHVXfvuaa3s5u/pub?gid=2028782809&single=true&output=csv
    - **Unnamed: 0:** Unused column. This column is created when creating this sub-dataset.
    - **Id:** Id of the records.
    - **SepalLengthCm:** Sepal length of flowers in cm
    - **SepalWidthCm:** Sepal width of flowers in cm

- **Petal All:** https://docs.google.com/spreadsheets/d/e/2PACX-1vQinLXShrOz4ExNaW1bSQVuvbbhIzJW7G0kkkD2SvqSD6STjLrQQiftgI7BGe10sBZi0CNr2_sJpQAz/pub?gid=1580010789&single=true&output=csv
    - **Unnamed: 0:** Unused column. This column is created when creating this sub-dataset.
    - **Id:** Id of the records.
    - **PetalLengthCm:** Petal length of flowers in cm
    - **PetalWidthCm:** Petal width of flowers in cm

- **Iris Virginica:** https://docs.google.com/spreadsheets/d/e/2PACX-1vSK39MwduGPHYNgw5yViezoLYCVDKMCWIHzjnt3GZNaxHPFOQLr2q6no_tyqTsOk-VfXleslfGVe9eJ/pub?gid=314231613&single=true&output=csv
    - **Unnamed: 0:** Unused column. This column is created when creating the sub-dataset.
    - **Id:** Id of the records.
    - **Species:** Name of this species.

- **Iris Versicolor:** https://docs.google.com/spreadsheets/d/e/2PACX-1vTcSFgLnabqIrgIc5WlwvnbbvyyJsgZjR-0E0-4TR-5aHgv_0EP6yNWglkkls3AXM2qHCR5VYzWCoTM/pub?gid=715607857&single=true&output=csv
    - **Unnamed: 0:** Unused column. This column is created when creating the sub-dataset.
    - **Id:** Id of the records.
    - **Species:** Name of this species.

- **Iris Setosa:** https://docs.google.com/spreadsheets/d/e/2PACX-1vSjqJpdgy2X_oDUUqQ0sSaFKqnnf8MYU4KgJSYgHaHmq0Wb1weMOsJXh-rICHmkLcaTkOwzMYLeh959/pub?gid=2003684803&single=true&output=csv
    - **Unnamed 0:** Unused column. This column is created when creating the sub-dataset.
    - **Id:** Id of the records.
    - **Species:** Name of this species.

In [58]:
import pandas as pd
sepal_all = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vT58ekmHTwptX7Bs4QOy6YByA1HMvYTACeeIjrKhHE0Pg1K_3egewHMKMh02zN9D5-yHVXfvuaa3s5u/pub?gid=2028782809&single=true&output=csv")
petal_all = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vQinLXShrOz4ExNaW1bSQVuvbbhIzJW7G0kkkD2SvqSD6STjLrQQiftgI7BGe10sBZi0CNr2_sJpQAz/pub?gid=1580010789&single=true&output=csv")

virginica = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vSK39MwduGPHYNgw5yViezoLYCVDKMCWIHzjnt3GZNaxHPFOQLr2q6no_tyqTsOk-VfXleslfGVe9eJ/pub?gid=314231613&single=true&output=csv")
versicolor = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vTcSFgLnabqIrgIc5WlwvnbbvyyJsgZjR-0E0-4TR-5aHgv_0EP6yNWglkkls3AXM2qHCR5VYzWCoTM/pub?gid=715607857&single=true&output=csv")
setosa = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vSjqJpdgy2X_oDUUqQ0sSaFKqnnf8MYU4KgJSYgHaHmq0Wb1weMOsJXh-rICHmkLcaTkOwzMYLeh959/pub?gid=2003684803&single=true&output=csv")


### `Q-9:` Plot a bar chart of the average Sepal Length  of Virginica and average Petal length of Setosa flower.

In [ ]:
# code here

### `Q-10:` Create the complete dataset by uisng the below datasets:
- virginica
- versicolor
- setosa
- sepal all
- petal all

This dataset should have these below column names in order:
1. Id
2. Species
3. SepalLengthCm
4. SepalWidthCm
5. PetalLengthCm
6. PetalWidthCm

Also, the dataset should be shuffled means the `Id` column should not be in increasing or decreasing order. So, make a dataset which has the shuffled Id column. You can use `DataFrame.sample()` method to shuffle. 

In [ ]:
# code here

### `Q-11:` Find out the maximum and minimum sepal width and petal width of Setosa and Versicolor. To do this:
- First create a dataset with merging the required datasets
- After that, use `groupby` to create groups based on the "Species" column.
- Then find out which are asked in this question.


The output should be like this:
```bash
Minimum Sepal width of Setosa is 2.3
Maximum Sepal width of Setosa is 4.4

**************************************************

Minimum Sepal width of Versicolor is 2.0
Maximum Sepal width of Versicolor is 3.4

**************************************************
```

In [ ]:
# code here